# Custom metrics, callbacks, and TensorBoard

Metrics carry state across a whole epoch; callbacks let you interrupt the loop without rewriting it.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 7 — A Deep Dive on Keras](../../../course-web-slides/ch07/index.html) &nbsp;·&nbsp; **Section:** 02 — Metrics, callbacks, and monitoring

---

## A metric is a stateful object

A loss is computed per batch and thrown away. A metric has to report a number over the **whole epoch**, so it accumulates.

In [ ]:
import keras
from keras import ops
import numpy as np

class RootMeanSquaredError(keras.metrics.Metric):
    def __init__(self, name="rmse", **kwargs):
        super().__init__(name=name, **kwargs)
        self.mse_sum = self.add_weight(shape=(), initializer="zeros",
                                       name="mse_sum")
        self.total_samples = self.add_weight(shape=(), initializer="zeros",
                                             name="total_samples",
                                             dtype="int32")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = ops.one_hot(y_true, num_classes=ops.shape(y_pred)[1])
        mse = ops.sum(ops.square(y_true - y_pred))
        self.mse_sum.assign_add(mse)
        num_samples = ops.shape(y_pred)[0]
        self.total_samples.assign_add(num_samples)

    def result(self):
        return ops.sqrt(self.mse_sum / ops.cast(self.total_samples, "float32"))

    def reset_state(self):
        self.mse_sum.assign(0.)
        self.total_samples.assign(0)

Three methods, and the third is the one people forget. Without `reset_state`, epoch two reports the running total from epoch one as well — a plausible number, quietly wrong.

## Using it

In [ ]:
from keras import layers
from keras.datasets import mnist

(x, y), (xt, yt) = mnist.load_data()
x = x.reshape(-1, 784).astype("float32") / 255
xt = xt.reshape(-1, 784).astype("float32") / 255

model = keras.Sequential([layers.Dense(64, activation="relu"),
                          layers.Dense(10, activation="softmax")])
model.compile(optimizer="rmsprop",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy", RootMeanSquaredError()])
model.fit(x, y, epochs=3, batch_size=128, validation_split=.2, verbose=2)

## The callbacks worth knowing

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=2,
        restore_best_weights=True,      # otherwise you keep the WORST weights
    ),
    keras.callbacks.ModelCheckpoint(
        filepath="checkpoint.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6,
    ),
]

model = keras.Sequential([layers.Dense(64, activation="relu"),
                          layers.Dense(10, activation="softmax")])
model.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
h = model.fit(x, y, epochs=30, batch_size=128, validation_split=.2,
              callbacks=callbacks, verbose=2)
print(f"stopped after {len(h.history['loss'])} of 30 epochs")

> ⚠️ **`restore_best_weights=True`.** Without it, `EarlyStopping` leaves you holding the weights from the *last* epoch — which, by definition of why it stopped, are the worst ones it saw.

## A callback of your own

In [ ]:
import matplotlib.pyplot as plt

class LossHistory(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.per_batch_losses = []

    def on_batch_end(self, batch, logs=None):
        self.per_batch_losses.append(logs["loss"])

    def on_epoch_end(self, epoch, logs=None):
        plt.figure(figsize=(7, 2.6))
        plt.plot(self.per_batch_losses, lw=.7)
        plt.title(f"per-batch loss through epoch {epoch}")
        plt.xlabel("batch"); plt.show()
        plt.close()

model = keras.Sequential([layers.Dense(64, activation="relu"),
                          layers.Dense(10, activation="softmax")])
model.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy")
model.fit(x[:12000], y[:12000], epochs=2, batch_size=128,
          callbacks=[LossHistory()], verbose=0)

Per-batch loss is far noisier than the per-epoch number `fit()` prints, and the noise is informative: a rising envelope means the learning rate is too high, long flat stretches mean it is too low.

## TensorBoard

In [ ]:
tb = keras.callbacks.TensorBoard(log_dir="./tb_logs")
model = keras.Sequential([layers.Dense(64, activation="relu"),
                          layers.Dense(10, activation="softmax")])
model.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(x, y, epochs=3, batch_size=128, validation_split=.2,
          callbacks=[tb], verbose=0)
print("now run:  tensorboard --logdir ./tb_logs")

Worth it once you are running more than a handful of experiments — which chapter 18 guarantees you will be.

---

## What to take away

- A metric is stateful: `update_state`, `result`, and **`reset_state`**.
- `EarlyStopping` without `restore_best_weights` leaves you the worst weights it saw.
- Callbacks hook the loop at batch, epoch, and train boundaries without rewriting it.
- Per-batch loss shows problems the per-epoch average hides.